# Adım 3b: Spark Structured Streaming — Kafka → Delta Lake Bronze
**Rojda sorumluluğu** — `feature/spark-eda` branch

> Docker servisleri çalışıyor olmalı: `docker-compose up -d`

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

KAFKA_BOOTSTRAP       = 'localhost:9092'
KAFKA_TOPIC           = 'climate-data'
BRONZE_STREAMING_PATH = './delta_lake/bronze_streaming'
CHECKPOINT_PATH       = './checkpoints/bronze_streaming'
STREAM_DURATION_SEC   = 120

MESSAGE_SCHEMA = StructType([
    StructField('timestamp',              StringType()),
    StructField('station_id',             StringType()),
    StructField('city_name',              StringType()),
    StructField('date',                   StringType()),
    StructField('season',                 StringType()),
    StructField('avg_temp_c',             DoubleType()),
    StructField('min_temp_c',             DoubleType()),
    StructField('max_temp_c',             DoubleType()),
    StructField('precipitation_mm',       DoubleType()),
    StructField('snow_depth_mm',          DoubleType()),
    StructField('avg_wind_speed_kmh',     DoubleType()),
    StructField('avg_sea_level_pres_hpa', DoubleType()),
    StructField('sunshine_total_min',     DoubleType()),
])

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateStructuredStreaming')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0,'
                'org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[STREAMING] Spark session hazir.')

26/05/12 14:49:40 WARN Utils: Your hostname, Canpolat-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/05/12 14:49:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/canpolat/.ivy2/cache
The jars for the packages stored in: /Users/canpolat/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-18fd7ab0-5950-45b1-b54d-6d4679db6b92;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central


:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 254ms :: artifacts dl 7ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	i

[STREAMING] Spark session hazir.


## Kafka → Delta Lake Streaming Sorgusu

In [2]:
def start_streaming(spark):
    print(f'[STREAMING] Kafka topic okunuyor: {KAFKA_TOPIC}')
    raw_stream = (
        spark.readStream
        .format('kafka')
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
        .option('subscribe', KAFKA_TOPIC)
        .option('startingOffsets', 'latest')
        .option('failOnDataLoss', 'false')
        .load()
    )
    parsed = (
        raw_stream
        .select(from_json(col('value').cast('string'), MESSAGE_SCHEMA).alias('d'))
        .select(
            col('d.station_id'), col('d.city_name'),
            to_date(col('d.date')).alias('date'), col('d.season'),
            col('d.avg_temp_c'), col('d.min_temp_c'), col('d.max_temp_c'),
            col('d.precipitation_mm'), col('d.snow_depth_mm'),
            col('d.avg_wind_speed_kmh'), col('d.avg_sea_level_pres_hpa'),
            col('d.sunshine_total_min'),
            current_timestamp().alias('ingested_at'),
        )
    )
    query = (
        parsed.writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_PATH)
        .option('path', BRONZE_STREAMING_PATH)
        .trigger(processingTime='10 seconds')
        .start()
    )
    print(f'[STREAMING] Hedef: {BRONZE_STREAMING_PATH}')
    print(f'[STREAMING] {STREAM_DURATION_SEC} saniye calisacak...')
    query.awaitTermination(STREAM_DURATION_SEC)
    query.stop()
    print('[STREAMING] Streaming tamamlandi.')
    try:
        count = spark.read.format('delta').load(BRONZE_STREAMING_PATH).count()
        print(f'[STREAMING] Bronze streaming: {count:,} kayit yazildi.')
    except Exception:
        print('[STREAMING] Bronze streaming tablosu bos veya henuz olusturulmadi.')

start_streaming(spark)
spark.stop()
print('Streaming pipeline tamamlandi.')

[STREAMING] Kafka topic okunuyor: climate-data


26/05/12 14:49:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/12 14:49:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


[STREAMING] Hedef: ./delta_lake/bronze_streaming
[STREAMING] 120 saniye calisacak...


26/05/12 14:49:46 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[STREAMING] Streaming tamamlandi.
[STREAMING] Bronze streaming: 3,727 kayit yazildi.
Streaming pipeline tamamlandi.
